# Notebook 31c — Wu 2003 S-B3: extended seed sweep and classification-threshold sensitivity

This notebook tests two cheap, no-new-data recommendations from the notebook 31b assessment:

1. **Extended seed sweep.** Notebook 24b only tried 5 seeds and picked seed 2 (which still failed
   SBC for `eta_col`/`xi_reb`, KS p=0.0001/0.008). SNPE training itself is fast (~15s/seed)
   regardless of how expensive the upstream bank generation was, so trying many more seeds is
   essentially free. This section trains seeds 5-19 (15 more) on the *same* mixed-wide bank nb24b
   used, and looks for a better-calibrated seed -- primarily for `xi_reb` (p=0.008, close to the
   0.05 line) since `eta_col`'s p=0.0001 looks structural (coverage gap) rather than noise, and pure
   seed variance is unlikely to fix a structural problem.

2. **Classification-threshold alignment.** Notebook 31b's `W2_cat_mild` failure (0/30 replicates
   correct) is a *classifier-definition* problem, not (only) a model problem: `sb3.py` uses
   `TRUE_UNIT_THRESH_SB3=0.90` for ground truth but `PRED_UNIT_THRESH_SB3=0.85` for prediction --
   inherited from notebook 31's propylene-oxide classifier, never re-derived for this plant. Any true
   value in `[0.85, 0.90)` (exactly where W2's `alpha=0.85` sits) is labeled `reactor` but needs an
   unusually low posterior estimate to be *predicted* `reactor`. This section re-runs the notebook
   31b classification loop at several prediction thresholds (0.85 current, 0.875, 0.90 aligned) for
   the best posterior found in Section 1, to see whether aligning the thresholds fixes `W2` without
   creating new problems elsewhere.

**Do not treat this notebook as changing recommendation 3** (widening `eta_col`'s prior further) --
that is a separate, larger intervention implemented in notebooks 22c/22d/24c/31d. This notebook only
tests whether recommendations 1-2 alone close enough of the gap to make that larger intervention
unnecessary.

In [1]:
import pathlib, pickle, sys, time, importlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats

cwd = pathlib.Path.cwd().resolve()
ROOT = cwd if (cwd / 'src').exists() else cwd.parent
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'scripts'))

from sbi.inference import SNPE
from sbi.neural_nets import posterior_nn
import cstr_sbi.recycle.sb3 as sb3
sb3 = importlib.reload(sb3)
from cstr_sbi.recycle.sb3 import (
    SB3_PARAM_NAMES, FAULT_UNITS_SB3, TRUE_UNIT_THRESH_SB3, PRED_UNIT_THRESH_SB3,
    mixed_wide_prior_4d_sb3, list_closed_loop_sb3, deterministic_window_sb3,
    fault_unit_sb3_truth, classify_fault_sb3,
)
from cstr_sbi.recycle.simulator import nominal_warm_start, deterministic_window, noisy_replicates
from cstr_sbi.recycle.summaries import compute_summaries, N_SUMMARIES_SB

DATA = ROOT / 'data'
RESULTS = ROOT / 'results'; RESULTS.mkdir(exist_ok=True)
FIGURES = ROOT / 'figures'; FIGURES.mkdir(exist_ok=True)
SBI_LOGS = ROOT / 'sbi-logs'; SBI_LOGS.mkdir(exist_ok=True)
OI = ['#000000', '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7']

# Same mixed-wide bank nb24b trained on -- no new data generated in this notebook.
MIXEDWIDE_TRAIN_PATH = DATA / 'wu2003_sbi_train_sb3_mixedwide_n5000.npz'
MIXEDWIDE_SBC_REDUCED_PATH = RESULTS / '24b_sysII_sb3_mixedwide_sbc_set_n120.npz'
MIXEDWIDE_SBC_CONFIRM_PATH = RESULTS / '24b_sysII_sb3_mixedwide_sbc_set_n400.npz'

N_POST = 200
ORIGINAL_SEEDS = [0, 1, 2, 3, 4]           # already trained by notebook 24b
EXTRA_SEEDS = list(range(5, 20))           # 15 additional seeds, this notebook's own contribution
ARCH, HIDDEN, TRANSFORMS = 'zuko_nsf', 60, 3
MAX_EPOCHS, STOP_AFTER_EPOCHS = 120, 12
y0_sb = nominal_warm_start('S-B')

def _load_bank(path, theta_key='thetas4'):
    if not path.exists():
        raise FileNotFoundError(f'{path.name} not found. Run notebook 22b (mixed-wide branch) first.')
    data = np.load(path)
    return np.asarray(data[theta_key], dtype=np.float32), np.asarray(data['summaries'], dtype=np.float32)

theta_train, summary_train = _load_bank(MIXEDWIDE_TRAIN_PATH)
theta_sbc_r, summary_sbc_r = _load_bank(MIXEDWIDE_SBC_REDUCED_PATH)
theta_sbc_c, summary_sbc_c = _load_bank(MIXEDWIDE_SBC_CONFIRM_PATH)
print(f'theta_train: {theta_train.shape}, theta_sbc_r: {theta_sbc_r.shape}, theta_sbc_c: {theta_sbc_c.shape}')
print(f'ORIGINAL_SEEDS={ORIGINAL_SEEDS} (from notebook 24b), EXTRA_SEEDS={EXTRA_SEEDS} (this notebook)')

theta_train: (5000, 4), theta_sbc_r: (120, 4), theta_sbc_c: (400, 4)
ORIGINAL_SEEDS=[0, 1, 2, 3, 4] (from notebook 24b), EXTRA_SEEDS=[5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19] (this notebook)


## Section 1 — Extended seed sweep

Reuses notebook 24b's `train_posterior_sb3`/`run_sbc_sb3` pattern verbatim (redefined locally since
each notebook in this project is self-contained), but sweeps 15 additional seeds on top of the 5
notebook 24b already tried. Existing per-seed posterior `.pkl` files are reused if present (so
re-running this notebook after a partial sweep does not retrain seeds already done).

In [2]:
def train_posterior_sb3(theta_train, summary_train, prior_4d, seed):
    torch.manual_seed(seed)
    inference = SNPE(prior=prior_4d, density_estimator=posterior_nn(model=ARCH, hidden_features=HIDDEN, num_transforms=TRANSFORMS))
    inference.append_simulations(torch.tensor(theta_train), torch.tensor(summary_train))
    t0 = time.time()
    trained = inference.train(max_num_epochs=MAX_EPOCHS, validation_fraction=0.1,
                              stop_after_epochs=STOP_AFTER_EPOCHS, show_train_summary=False)
    return inference.build_posterior(trained), time.time() - t0


def run_sbc_sb3(posterior, theta_set, summary_set, n_post, seed, label):
    torch.manual_seed(seed)
    ranks = {name: [] for name in SB3_PARAM_NAMES}
    hits90 = {name: 0 for name in SB3_PARAM_NAMES}
    widths90 = {name: [] for name in SB3_PARAM_NAMES}
    for th, s in zip(theta_set, summary_set):
        with torch.no_grad():
            samp = posterior.sample((n_post,), x=torch.tensor(s, dtype=torch.float32),
                                    show_progress_bars=False, reject_outside_prior=False).detach().cpu().numpy()
        for k, name in enumerate(SB3_PARAM_NAMES):
            ranks[name].append(int(np.sum(samp[:, k] < th[k])))
            lo, hi = np.percentile(samp[:, k], [5, 95])
            widths90[name].append(float(hi - lo))
            if lo <= th[k] <= hi:
                hits90[name] += 1
    out = {'n_sbc': int(len(theta_set)), 'n_post': int(n_post)}
    for name in SB3_PARAM_NAMES:
        r = np.asarray(ranks[name], dtype=float)
        ks = scipy_stats.ks_1samp(r / n_post, scipy_stats.uniform.cdf)
        out[name] = {'ks_pvalue': float(ks.pvalue), 'mean_rank_frac': float(r.mean() / n_post),
                     'coverage_90': float(hits90[name] / len(theta_set)),
                     'mean_ci_width_90': float(np.mean(widths90[name]))}
    out['min_ks_pvalue'] = min(out[name]['ks_pvalue'] for name in SB3_PARAM_NAMES)
    out['max_abs_rank_bias'] = max(abs(out[name]['mean_rank_frac'] - 0.5) for name in SB3_PARAM_NAMES)
    return out


def flatten_run(seed, result):
    row = {'seed': seed, 'min_ks_pvalue': result['min_ks_pvalue'], 'max_abs_rank_bias': result['max_abs_rank_bias']}
    for name in SB3_PARAM_NAMES:
        row[f'{name}_ks_pvalue'] = result[name]['ks_pvalue']
        row[f'{name}_mean_rank_frac'] = result[name]['mean_rank_frac']
        row[f'{name}_coverage_90'] = result[name]['coverage_90']
    return row


prior_4d = mixed_wide_prior_4d_sb3(uniform_idx=(0, 1))  # same prior nb24b used -- no new coverage here
all_seeds = ORIGINAL_SEEDS + EXTRA_SEEDS
posteriors = {}
confirm_rows = []
t_sweep0 = time.time()
for seed in all_seeds:
    posterior_path = SBI_LOGS / f'wu2003_posterior_sb3_seed{seed}.pkl'
    if posterior_path.exists():
        with open(posterior_path, 'rb') as f:
            payload = pickle.load(f)
        posterior = payload['posterior']
        print(f'loaded existing {posterior_path.name}')
    else:
        posterior, train_wall_s = train_posterior_sb3(theta_train, summary_train, prior_4d, seed)
        with open(posterior_path, 'wb') as f:
            pickle.dump({'posterior': posterior, 'train_wall_s': train_wall_s, 'seed': seed,
                         'structure': 'S-B3', 'bank': str(MIXEDWIDE_TRAIN_PATH)}, f)
        print(f'trained + saved {posterior_path.name} in {train_wall_s:.1f}s')
    posteriors[seed] = posterior
    confirm = run_sbc_sb3(posterior, theta_sbc_c, summary_sbc_c, n_post=N_POST, seed=248000 + seed, label=f'confirm seed{seed}')
    confirm_rows.append(flatten_run(seed, confirm))
    print(f'  seed {seed}: min_ks_pvalue={confirm["min_ks_pvalue"]:.4g}')

confirm_df = pd.DataFrame(confirm_rows)
confirm_df.to_csv(RESULTS / '31c_extended_seed_sweep_confirm_sbc.csv', index=False)
print(f'\nswept {len(all_seeds)} seeds total in {time.time() - t_sweep0:.0f}s')

loaded existing wu2003_posterior_sb3_seed0.pkl


/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 11.5% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 38.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 10.5% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 8.5% of samples drawn

  seed 0: min_ks_pvalue=1.777e-05
loaded existing wu2003_posterior_sb3_seed1.pkl


/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 11.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 8.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 27.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 13.5% of samples drawn

  seed 1: min_ks_pvalue=1.121e-18
loaded existing wu2003_posterior_sb3_seed2.pkl


/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 45.5% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 9.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 29.5% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 15.5% of samples drawn

... (32 outputs left out)

  seed 18: min_ks_pvalue=3.968e-06
 Neural network successfully converged after 94 epochs.trained + saved wu2003_posterior_sb3_seed19.pkl in 14.9s


/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 44.5% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 33.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 66.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 43.5% of samples draw

  seed 19: min_ks_pvalue=8.438e-14

swept 20 seeds total in 317s


/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/560054551.py:18: UserWarning: 57.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = posterior.sample((n_post,), x=torch.tensor(s, dtype=torch.float32),


## Section 2 — Compare extended sweep vs. notebook 24b's original 5-seed result

Notebook 24b selected seed 2 (`min_ks_pvalue=7.3e-05`, driven by `eta_col`). This section reports
whether any of the 15 new seeds does better, and specifically whether any seed pushes `xi_reb`
(p=0.008 for seed 2) over the 0.05 line -- `eta_col`'s p=0.0001 is expected to stay near-zero
regardless of seed, since that failure is diagnosed as a coverage problem, not a training-noise
problem.

In [3]:
display_cols = ['seed', 'min_ks_pvalue', 'max_abs_rank_bias'] + [f'{name}_ks_pvalue' for name in SB3_PARAM_NAMES]
ranked = confirm_df.sort_values(['min_ks_pvalue', 'max_abs_rank_bias'], ascending=[False, True])
print('All seeds, ranked by confirm-budget min KS p-value:')
print(ranked[display_cols].to_string(index=False))

best_seed = int(ranked.iloc[0]['seed'])
best_row = ranked.iloc[0]
print(f'\nBest seed overall: {best_seed} (nb24b originally picked seed 2)')
print('Per-parameter calibration for the best seed:')
for name in SB3_PARAM_NAMES:
    print(f"  {name:8s} KS p={best_row[f'{name}_ks_pvalue']:.4g}  mean_rank={best_row[f'{name}_mean_rank_frac']:.3f}  cov90={best_row[f'{name}_coverage_90']:.3f}")

n_pass_alpha = int((confirm_df['alpha_ks_pvalue'] > 0.05).sum())
n_pass_betar = int((confirm_df['beta_r_ks_pvalue'] > 0.05).sum())
n_pass_eta = int((confirm_df['eta_col_ks_pvalue'] > 0.05).sum())
n_pass_xi = int((confirm_df['xi_reb_ks_pvalue'] > 0.05).sum())
print(f'\nSeeds passing KS>0.05 out of {len(all_seeds)}: alpha={n_pass_alpha}, beta_r={n_pass_betar}, eta_col={n_pass_eta}, xi_reb={n_pass_xi}')
print('If eta_col/xi_reb pass counts are ~0, that confirms the coverage-gap diagnosis (seed variance alone cannot fix it).')

best_posterior = posteriors[best_seed]

All seeds, ranked by confirm-budget min KS p-value:
 seed  min_ks_pvalue  max_abs_rank_bias  alpha_ks_pvalue  beta_r_ks_pvalue  eta_col_ks_pvalue  xi_reb_ks_pvalue
    5   6.191797e-04           0.038813     6.191797e-04      3.779122e-02       5.308901e-01      9.220330e-04
    2   7.262969e-05           0.053850     3.814607e-01      3.174450e-01       7.262969e-05      8.130082e-03
    0   1.777190e-05           0.081650     1.777190e-05      5.308901e-01       1.127992e-02      1.549314e-02
   13   1.777190e-05           0.071500     1.777190e-05      2.106671e-02       2.835831e-02      2.612124e-01
    4   1.089165e-05           0.052075     1.089165e-05      5.308901e-01       4.097587e-03      6.511749e-02
   18   3.967534e-06           0.056800     2.835831e-02      1.127992e-02       3.967534e-06      9.220330e-04
   11   8.079292e-07           0.082487     2.870440e-05      2.126329e-01       2.358199e-06      8.079292e-07
   12   2.656868e-07           0.087888     2.656868

## Section 3 — Classification-threshold sensitivity

Re-runs the notebook 31b classification loop (12 retained scenarios x 30 noisy replicates) for the
best posterior found in Section 2, at three prediction thresholds: the current 0.85, an intermediate
0.875, and 0.90 (fully aligned with `TRUE_UNIT_THRESH_SB3`). Ground truth always uses 0.90 -- only
the *prediction* threshold is varied, since that is the one inherited from notebook 31's
propylene-oxide classifier without being re-derived for this plant.

In [4]:
def predict_unit_sb3(theta_row, thresh):
    """Mirrors `fault_unit_sb3_predict` in sb3.py, but with a configurable threshold
    instead of the hardcoded `PRED_UNIT_THRESH_SB3=0.85` default."""
    alpha, beta_r, eta_col, xi_reb = np.asarray(theta_row, dtype=np.float32)[:4]
    reactor_bad = (alpha < thresh) or (beta_r < thresh)
    column_bad = (eta_col < thresh) or (xi_reb < thresh)
    n_bad = int(reactor_bad) + int(column_bad)
    if n_bad == 0:
        return 'healthy'
    if n_bad >= 2:
        return 'multi'
    return 'reactor' if reactor_bad else 'column'


def classify_with_threshold(samples, thresh):
    labels = [predict_unit_sb3(row, thresh) for row in samples]
    probs = {u: float(np.mean([l == u for l in labels])) for u in FAULT_UNITS_SB3}
    return max(probs, key=probs.__getitem__), probs


def compute_metrics(preds, trues, classes=FAULT_UNITS_SB3):
    cm = np.zeros((len(classes), len(classes)), dtype=int)
    c2i = {c: i for i, c in enumerate(classes)}
    for true, pred in zip(trues, preds):
        cm[c2i[true], c2i[pred]] += 1
    f1 = {}
    for i, cls in enumerate(classes):
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1[cls] = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
    return {'confusion_matrix': cm, 'per_class_f1': f1, 'macro_f1': float(np.mean(list(f1.values()))), 'accuracy': float(np.trace(cm) / cm.sum())}


N_REPS = 30
PRED_THRESHOLDS_TO_TEST = [0.85, 0.875, 0.90]
threshold_results = {}

t0 = time.time()
for pred_thresh in PRED_THRESHOLDS_TO_TEST:
    all_preds, all_trues = [], []
    scenario_rows = []
    for sc in list_closed_loop_sb3():
        t_h, raw_obs = deterministic_window(sc, structure='S-B', y0=y0_sb)
        t_h_arr = np.asarray(t_h)
        rng = np.random.default_rng(311001 + sc.id)
        reps = noisy_replicates(np.asarray(raw_obs), n_replicates=N_REPS, rng=rng, noise_pct=0.003)
        true_unit = fault_unit_sb3_truth(np.asarray(sc.theta())[:4])
        rep_preds = []
        for i in range(N_REPS):
            s = compute_summaries(reps[i], 'S-B', t_h_arr)
            with torch.no_grad():
                samp = best_posterior.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
                                             show_progress_bars=False, reject_outside_prior=False).detach().cpu().numpy()
            pred_class, _ = classify_with_threshold(samp, pred_thresh)
            rep_preds.append(pred_class)
            all_preds.append(pred_class)
            all_trues.append(true_unit)
        scenario_rows.append({'scenario': sc.name, 'true_unit': true_unit,
                              'scenario_accuracy': float(np.mean([p == true_unit for p in rep_preds]))})
    m = compute_metrics(all_preds, all_trues)
    threshold_results[pred_thresh] = {'metrics': m, 'scenario_df': pd.DataFrame(scenario_rows)}
    print(f'pred_thresh={pred_thresh}: accuracy={m["accuracy"]:.3f}, macro_f1={m["macro_f1"]:.3f}, per_class_f1={m["per_class_f1"]}')
print(f'\ndone in {time.time() - t0:.0f}s')

/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/1027033034.py:55: UserWarning: 10.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = best_posterior.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/1027033034.py:55: UserWarning: 32.5% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = best_posterior.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/1027033034.py:55: UserWarning: 28.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = best_posterior.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/1027033034.py:55: UserWarning: 6.

pred_thresh=0.85: accuracy=0.833, macro_f1=0.798, per_class_f1={'healthy': np.float64(0.6666666666666666), 'reactor': np.float64(0.8571428571428571), 'column': np.float64(1.0), 'multi': np.float64(0.6666666666666666)}


/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/1027033034.py:55: UserWarning: 11.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = best_posterior.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/1027033034.py:55: UserWarning: 8.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = best_posterior.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/1027033034.py:55: UserWarning: 42.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = best_posterior.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/1027033034.py:55: UserWarning: 25.

pred_thresh=0.875: accuracy=0.975, macro_f1=0.968, per_class_f1={'healthy': np.float64(0.9375), 'reactor': np.float64(0.9786223277909738), 'column': np.float64(1.0), 'multi': np.float64(0.9565217391304348)}


/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/1027033034.py:55: UserWarning: 12.5% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = best_posterior.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/1027033034.py:55: UserWarning: 8.0% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = best_posterior.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/1027033034.py:55: UserWarning: 26.5% of samples drawn with reject_outside_prior=False lie outside the prior support. This may lead to incorrect inference.
  samp = best_posterior.sample((N_POST,), x=torch.tensor(s, dtype=torch.float32),
/var/folders/pk/y59h45m94tq3dzsk9n26k2kc0000gp/T/ipykernel_6827/1027033034.py:55: UserWarning: 22.

pred_thresh=0.9: accuracy=1.000, macro_f1=1.000, per_class_f1={'healthy': np.float64(1.0), 'reactor': np.float64(1.0), 'column': np.float64(1.0), 'multi': np.float64(1.0)}

done in 399s


In [5]:
for pred_thresh, res in threshold_results.items():
    print(f'=== pred_thresh={pred_thresh} ===')
    print(res['scenario_df'][res['scenario_df']['scenario_accuracy'] < 1.0].to_string(index=False))
    print()

=== pred_thresh=0.85 ===
             scenario true_unit  scenario_accuracy
          W2_cat_mild   reactor                0.0
W12_snowball_compound     multi                0.0

=== pred_thresh=0.875 ===
             scenario true_unit  scenario_accuracy
          W2_cat_mild   reactor           0.866667
W12_snowball_compound     multi           0.833333

=== pred_thresh=0.9 ===
Empty DataFrame
Columns: [scenario, true_unit, scenario_accuracy]
Index: []



## Section 4 — Assessment and recommendation

Summarizes the best (seed, threshold) combination found and whether it clears notebook 31b's
acceptance bar without any new data generation.

In [6]:
best_thresh = max(threshold_results, key=lambda t: threshold_results[t]['metrics']['macro_f1'])
best_thresh_metrics = threshold_results[best_thresh]['metrics']
print('=== Assessment ===')
print(f'Best seed: {best_seed} (nb24b originally used seed 2)')
print(f'Best prediction threshold: {best_thresh} (current default is 0.85)')
print(f'Resulting accuracy={best_thresh_metrics["accuracy"]:.3f}, macro_f1={best_thresh_metrics["macro_f1"]:.3f}')
print(f'Per-class F1: {best_thresh_metrics["per_class_f1"]}')
print()
print('Compare to notebook 31b baseline: accuracy=0.831, macro_f1=0.793 (seed 2, pred_thresh=0.85).')
print('If macro_f1 improved meaningfully and cleared 0.85, recommendations 1-2 alone may be sufficient')
print('and the larger eta_col-widening intervention (notebooks 22c/22d/24c/31d) may not be needed.')
print('If eta_col-driven failures (e.g. W12_snowball_compound) persist regardless of seed/threshold,')
print('that confirms the coverage-gap diagnosis and motivates proceeding with the widening.')

=== Assessment ===
Best seed: 5 (nb24b originally used seed 2)
Best prediction threshold: 0.9 (current default is 0.85)
Resulting accuracy=1.000, macro_f1=1.000
Per-class F1: {'healthy': np.float64(1.0), 'reactor': np.float64(1.0), 'column': np.float64(1.0), 'multi': np.float64(1.0)}

Compare to notebook 31b baseline: accuracy=0.831, macro_f1=0.793 (seed 2, pred_thresh=0.85).
If macro_f1 improved meaningfully and cleared 0.85, recommendations 1-2 alone may be sufficient
and the larger eta_col-widening intervention (notebooks 22c/22d/24c/31d) may not be needed.
If eta_col-driven failures (e.g. W12_snowball_compound) persist regardless of seed/threshold,
that confirms the coverage-gap diagnosis and motivates proceeding with the widening.


## Final acceptance criteria

This notebook's exploration is judged a sufficient fix (making the larger data-generation
intervention unnecessary) if, for the best (seed, threshold) combination found:
1. accuracy is at least 0.85,
2. macro-F1 is at least 0.85,
3. every retained fault unit has F1 at least 0.70,
4. no retained scenario has pooled/replicate accuracy below 0.50.

In [7]:
best_scenario_df = threshold_results[best_thresh]['scenario_df']
criteria = [
    ('accuracy >= 0.85', float(best_thresh_metrics['accuracy']) >= 0.85, float(best_thresh_metrics['accuracy'])),
    ('macro-F1 >= 0.85', float(best_thresh_metrics['macro_f1']) >= 0.85, float(best_thresh_metrics['macro_f1'])),
    ('all retained class F1 >= 0.70', bool(all(v >= 0.70 for v in best_thresh_metrics['per_class_f1'].values())), best_thresh_metrics['per_class_f1']),
    ('no scenario accuracy below 0.50', bool((best_scenario_df['scenario_accuracy'] >= 0.50).all()), best_scenario_df[['scenario', 'scenario_accuracy']].to_dict('records')),
]
for name, passed, value in criteria:
    print(f"{'PASS' if passed else 'FAIL'} | {name}: {value}")

if all(passed for _, passed, _ in criteria):
    print('\n=> Recommendations 1-2 alone appear sufficient; the eta_col-widening intervention (22c/22d/24c/31d) is optional.')
else:
    print('\n=> Gap remains; proceed with the eta_col-widening intervention (22c/22d/24c/31d).')

PASS | accuracy >= 0.85: 1.0
PASS | macro-F1 >= 0.85: 1.0
PASS | all retained class F1 >= 0.70: {'healthy': np.float64(1.0), 'reactor': np.float64(1.0), 'column': np.float64(1.0), 'multi': np.float64(1.0)}
PASS | no scenario accuracy below 0.50: [{'scenario': 'W1_healthy', 'scenario_accuracy': 1.0}, {'scenario': 'W2_cat_mild', 'scenario_accuracy': 1.0}, {'scenario': 'W3_cat_severe', 'scenario_accuracy': 1.0}, {'scenario': 'W4_cat_threshold', 'scenario_accuracy': 1.0}, {'scenario': 'W5_jacket_mild', 'scenario_accuracy': 1.0}, {'scenario': 'W6_jacket_severe', 'scenario_accuracy': 1.0}, {'scenario': 'W7_col_eff_mild', 'scenario_accuracy': 1.0}, {'scenario': 'W9_reb_fouling', 'scenario_accuracy': 1.0}, {'scenario': 'W11_reactor_combined', 'scenario_accuracy': 1.0}, {'scenario': 'W12_snowball_compound', 'scenario_accuracy': 1.0}, {'scenario': 'W15_snowball_threshold', 'scenario_accuracy': 1.0}, {'scenario': 'W16_full_multi', 'scenario_accuracy': 1.0}]

=> Recommendations 1-2 alone appear su